# Demo 1 — End-to-End Validation

This notebook validates the completed batch and streaming Bronze pipelines.

It checks:

- historical Bronze table existence
- streaming Bronze table existence
- expected symbols
- duplicate business keys
- duplicate event IDs
- null critical fields
- historical date coverage
- streaming event coverage
- schema evolution columns
- Event Hub metadata columns
- checkpoint accessibility
- external Volume folders
- source file availability

The notebook finishes with a clear PASS/FAIL summary.


## 1. Load shared configuration

All table names, Volume paths, symbols, and checkpoint paths come from `config/00_config`.


In [0]:
%run ../config/00_config


## 2. Import required Spark functions


In [0]:
from pyspark.sql import functions as F


## 3. Define validation helpers

Each validation produces:

- validation name
- status
- details

Critical failures are collected and reported at the end.


In [0]:
validation_results = []


def add_result(
    validation_name: str,
    passed: bool,
    details: str,
) -> None:
    validation_results.append({
        "validation_name": validation_name,
        "status": "PASS" if passed else "FAIL",
        "details": details,
    })


def table_exists(table_name: str) -> bool:
    return spark.catalog.tableExists(table_name)


## 4. Validate required Unity Catalog tables

The expected permanent Bronze tables are:

- historical OHLCV Bronze
- streaming crypto ticks Bronze


In [0]:
historical_table_exists = table_exists(
    historical_bronze_table
)

streaming_table_exists = table_exists(
    streaming_bronze_table
)

add_result(
    "Historical Bronze table exists",
    historical_table_exists,
    historical_bronze_table,
)

add_result(
    "Streaming Bronze table exists",
    streaming_table_exists,
    streaming_bronze_table,
)

if not historical_table_exists:
    raise RuntimeError(
        f"Historical Bronze table is missing: "
        f"{historical_bronze_table}"
    )

if not streaming_table_exists:
    raise RuntimeError(
        f"Streaming Bronze table is missing: "
        f"{streaming_bronze_table}"
    )


## 5. Load the permanent Bronze tables


In [0]:
historical_df = spark.table(
    historical_bronze_table
)

streaming_df = spark.table(
    streaming_bronze_table
)

print(
    f"Historical rows: {historical_df.count()}"
)

print(
    f"Streaming rows: {streaming_df.count()}"
)


## 6. Validate historical symbols and row counts

For January 2026, each symbol should normally have about 31 daily rows.


In [0]:
historical_symbol_summary_df = (
    historical_df
    .groupBy("symbol")
    .agg(
        F.count("*").alias("row_count"),
        F.min("open_time").alias("first_open_time"),
        F.max("open_time").alias("last_open_time"),
    )
    .orderBy("symbol")
)

display(historical_symbol_summary_df)

actual_historical_symbols = {
    row["symbol"]
    for row in historical_symbol_summary_df.collect()
}

expected_historical_symbols = set(
    historical_symbols
)

symbols_match = (
    actual_historical_symbols
    == expected_historical_symbols
)

add_result(
    "Historical symbols match configuration",
    symbols_match,
    (
        f"Expected: {sorted(expected_historical_symbols)}; "
        f"Actual: {sorted(actual_historical_symbols)}"
    ),
)

historical_total_rows = historical_df.count()

add_result(
    "Historical table contains data",
    historical_total_rows > 0,
    f"Total rows: {historical_total_rows}",
)


## 7. Validate historical business-key uniqueness

The unique business key is:

`symbol + open_time`


In [0]:
historical_duplicate_keys_df = (
    historical_df
    .groupBy(
        "symbol",
        "open_time",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

historical_duplicate_count = (
    historical_duplicate_keys_df.count()
)

add_result(
    "Historical business keys are unique",
    historical_duplicate_count == 0,
    (
        f"Duplicate symbol + open_time groups: "
        f"{historical_duplicate_count}"
    ),
)

if historical_duplicate_count > 0:
    display(historical_duplicate_keys_df)


## 8. Validate historical critical fields

Critical historical fields must not be null.


In [0]:
historical_invalid_df = (
    historical_df
    .filter(
        F.col("record_id").isNull()
        | F.col("symbol").isNull()
        | F.col("open_time").isNull()
        | F.col("close_time").isNull()
        | F.col("open").isNull()
        | F.col("high").isNull()
        | F.col("low").isNull()
        | F.col("close").isNull()
        | F.col("source_system").isNull()
        | F.col("ingested_at").isNull()
    )
)

historical_invalid_count = (
    historical_invalid_df.count()
)

add_result(
    "Historical critical fields are populated",
    historical_invalid_count == 0,
    f"Invalid historical rows: {historical_invalid_count}",
)

if historical_invalid_count > 0:
    display(historical_invalid_df)


## 9. Validate OHLC business rules

For each daily candle:

- `high` should be greater than or equal to open, low, and close
- `low` should be less than or equal to open, high, and close
- volume should not be negative


In [0]:
historical_ohlc_invalid_df = (
    historical_df
    .filter(
        (F.col("high") < F.col("open"))
        | (F.col("high") < F.col("low"))
        | (F.col("high") < F.col("close"))
        | (F.col("low") > F.col("open"))
        | (F.col("low") > F.col("high"))
        | (F.col("low") > F.col("close"))
        | (F.col("volume") < 0)
    )
)

historical_ohlc_invalid_count = (
    historical_ohlc_invalid_df.count()
)

add_result(
    "Historical OHLC rules are valid",
    historical_ohlc_invalid_count == 0,
    (
        f"Rows violating OHLC rules: "
        f"{historical_ohlc_invalid_count}"
    ),
)

if historical_ohlc_invalid_count > 0:
    display(historical_ohlc_invalid_df)


## 10. Validate streaming symbols and event counts

The streaming table should contain BTC, ETH, and SOL events.


In [0]:
streaming_symbol_summary_df = (
    streaming_df
    .groupBy("symbol")
    .agg(
        F.count("*").alias("event_count"),
        F.min("event_time").alias("first_event_time"),
        F.max("event_time").alias("last_event_time"),
        F.min("price_usd").alias("minimum_price"),
        F.max("price_usd").alias("maximum_price"),
    )
    .orderBy("symbol")
)

display(streaming_symbol_summary_df)

actual_streaming_symbols = {
    row["symbol"]
    for row in streaming_symbol_summary_df.collect()
}

expected_streaming_symbols = set(
    historical_symbols
)

streaming_symbols_match = (
    actual_streaming_symbols
    == expected_streaming_symbols
)

add_result(
    "Streaming symbols match configuration",
    streaming_symbols_match,
    (
        f"Expected: {sorted(expected_streaming_symbols)}; "
        f"Actual: {sorted(actual_streaming_symbols)}"
    ),
)

streaming_total_rows = streaming_df.count()

add_result(
    "Streaming table contains data",
    streaming_total_rows > 0,
    f"Total rows: {streaming_total_rows}",
)


## 11. Validate streaming event ID uniqueness

`event_id` should uniquely identify each produced event.


In [0]:
streaming_duplicate_event_ids_df = (
    streaming_df
    .groupBy("event_id")
    .count()
    .filter(
        F.col("count") > 1
    )
)

streaming_duplicate_event_id_count = (
    streaming_duplicate_event_ids_df.count()
)

add_result(
    "Streaming event IDs are unique",
    streaming_duplicate_event_id_count == 0,
    (
        f"Duplicate event ID groups: "
        f"{streaming_duplicate_event_id_count}"
    ),
)

if streaming_duplicate_event_id_count > 0:
    display(streaming_duplicate_event_ids_df)


## 12. Validate streaming critical fields

Critical streaming fields must not be null, and prices must be positive.


In [0]:
streaming_invalid_df = (
    streaming_df
    .filter(
        F.col("event_id").isNull()
        | F.col("symbol").isNull()
        | F.col("price_usd").isNull()
        | (F.col("price_usd") <= 0)
        | F.col("event_time").isNull()
        | F.col("producer_id").isNull()
        | F.col("source_system").isNull()
        | F.col("eventhub_partition").isNull()
        | F.col("eventhub_offset").isNull()
        | F.col("eventhub_enqueued_at").isNull()
        | F.col("raw_json").isNull()
        | F.col("ingested_at").isNull()
    )
)

streaming_invalid_count = (
    streaming_invalid_df.count()
)

add_result(
    "Streaming critical fields are populated",
    streaming_invalid_count == 0,
    f"Invalid streaming rows: {streaming_invalid_count}",
)

if streaming_invalid_count > 0:
    display(streaming_invalid_df)


## 13. Validate Event Hub partition and offset uniqueness

The combination of partition and offset should be unique for each Event Hub record.


In [0]:
eventhub_duplicate_offsets_df = (
    streaming_df
    .groupBy(
        "eventhub_partition",
        "eventhub_offset",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

eventhub_duplicate_offset_count = (
    eventhub_duplicate_offsets_df.count()
)

add_result(
    "Event Hub partition-offset keys are unique",
    eventhub_duplicate_offset_count == 0,
    (
        f"Duplicate partition-offset groups: "
        f"{eventhub_duplicate_offset_count}"
    ),
)

if eventhub_duplicate_offset_count > 0:
    display(eventhub_duplicate_offsets_df)


## 14. Validate schema evolution columns

The evolved Bronze table should contain these four additional columns:

- `change_pct_24h`
- `high_price_24h`
- `low_price_24h`
- `volume_24h`


In [0]:
required_evolution_columns = {
    "change_pct_24h",
    "high_price_24h",
    "low_price_24h",
    "volume_24h",
}

actual_streaming_columns = set(
    streaming_df.columns
)

missing_evolution_columns = (
    required_evolution_columns
    - actual_streaming_columns
)

add_result(
    "Schema evolution columns exist",
    not missing_evolution_columns,
    (
        "Missing columns: "
        f"{sorted(missing_evolution_columns)}"
        if missing_evolution_columns
        else "All expected evolution columns exist"
    ),
)


## 15. Validate evolved and original event coexistence

After schema evolution:

- old events should have nulls in the new fields
- evolved events should contain values in the new fields


In [0]:
if not missing_evolution_columns:
    evolved_event_count = (
        streaming_df
        .filter(
            F.col("change_pct_24h").isNotNull()
        )
        .count()
    )

    original_event_count = (
        streaming_df
        .filter(
            F.col("change_pct_24h").isNull()
        )
        .count()
    )

    add_result(
        "Evolved streaming events exist",
        evolved_event_count > 0,
        f"Evolved events: {evolved_event_count}",
    )

    add_result(
        "Original streaming events remain",
        original_event_count > 0,
        f"Original events: {original_event_count}",
    )
else:
    add_result(
        "Evolved streaming events exist",
        False,
        "Cannot validate because evolution columns are missing",
    )

    add_result(
        "Original streaming events remain",
        False,
        "Cannot validate because evolution columns are missing",
    )


## 16. Validate source files in the external Volume

The historical source folder should contain the three configured Binance CSV files.


In [0]:
expected_source_files = {
    f"{symbol}-{historical_interval}-"
    f"{historical_period}.csv"
    for symbol in historical_symbols
}

actual_source_files = {
    file_info.name
    for file_info in dbutils.fs.ls(
        historical_raw_path
    )
    if file_info.name.lower().endswith(
        ".csv"
    )
}

missing_source_files = (
    expected_source_files
    - actual_source_files
)

add_result(
    "Historical source files exist",
    not missing_source_files,
    (
        f"Missing files: {sorted(missing_source_files)}"
        if missing_source_files
        else f"Files found: {sorted(actual_source_files)}"
    ),
)


## 17. Validate Volume folders and checkpoint accessibility

The configured folders should be accessible through the external Volume.


In [0]:
paths_to_validate = {
    "Volume root": volume_root,
    "Historical raw path": historical_raw_path,
    "Streaming test path": streaming_test_path,
    "Historical landing path": historical_landing_path,
    "Schema path": crypto_ticks_schema_path,
    "Checkpoint path": crypto_ticks_checkpoint_path,
    "Archive path": archive_path,
}

for path_name, path_value in paths_to_validate.items():
    try:
        dbutils.fs.ls(path_value)

        add_result(
            f"{path_name} is accessible",
            True,
            path_value,
        )
    except Exception as exc:
        add_result(
            f"{path_name} is accessible",
            False,
            f"{path_value}: {exc}",
        )


## 18. Display a combined data preview

This section shows the historical candles and the latest streaming events side by side as separate outputs.


In [0]:
display(
    historical_df
    .select(
        "symbol",
        "open_time",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "source_system",
    )
    .orderBy(
        "symbol",
        "open_time",
    )
)

display(
    streaming_df
    .select(
        "symbol",
        "price_usd",
        "event_time",
        "change_pct_24h",
        "high_price_24h",
        "low_price_24h",
        "volume_24h",
        "eventhub_partition",
        "eventhub_offset",
    )
    .orderBy(
        F.col("event_time").desc()
    )
)


## 19. Final validation summary

All checks are displayed in one table.

If any validation fails, the notebook raises an error so it can also be used as a workflow validation task.


In [0]:
validation_results_df = (
    spark.createDataFrame(
        validation_results
    )
    .orderBy(
        F.col("status").asc(),
        F.col("validation_name").asc(),
    )
)

display(validation_results_df)

failed_validations = [
    result
    for result in validation_results
    if result["status"] == "FAIL"
]

passed_count = sum(
    1
    for result in validation_results
    if result["status"] == "PASS"
)

failed_count = len(
    failed_validations
)

print("Demo 1 validation summary")
print("-------------------------")
print(f"Passed: {passed_count}")
print(f"Failed: {failed_count}")
print(
    f"Total checks: "
    f"{len(validation_results)}"
)

if failed_validations:
    failed_names = [
        result["validation_name"]
        for result in failed_validations
    ]

    raise RuntimeError(
        "Demo 1 validation failed: "
        f"{failed_names}"
    )

print("All Demo 1 validations passed successfully.")


## Board explanation

> The validation notebook checks both Bronze pipelines independently. It verifies completeness, uniqueness, required metadata, Event Hub offsets, schema evolution, source files, checkpoints, and external Volume accessibility. The notebook fails automatically when a critical check does not pass.

## Next optional notebooks

- `transformation/08_silver_transformation.ipynb`
- `transformation/09_gold_aggregation.ipynb`

If the project scope remains Bronze-only, the next step can instead be the dashboard queries and Databricks workflow definition.
